# Task 6 - Operational versus Commercial Diagnosis

Four stakeholders each proposed an explanation. Each gets a test that **holds
the mix constant and then looks at the rate** - the only way to tell a real
performance change from a composition change.

Three of the four turn out to be wrong, and one of them is wrong because of a
data-quality artefact rather than a business fact.

Put this notebook in `06_Operational_commercial_diagnosis/`.

In [23]:
"""Task 6 - operational versus commercial diagnosis.

Each stakeholder hypothesis gets a test that HOLDS THE MIX CONSTANT and then
looks at the rate. That is the only way to tell a real performance change from
a composition change, and it is where three of the four stakeholders are wrong.

Every test returns a table plus a one-line verdict.
"""
import numpy as np
import pandas as pd

INCIDENT_MONTHS = ["2025-08", "2025-09"]
MODEL_CUTOVER_MONTH = "2025-09"
DECLINE_THRESHOLD = 0.80     # on the normalised 0-1 scale

from pathlib import Path

CANDIDATES = [
    Path("model_output/fact_transaction.csv"),
    Path("../04_profitability_model/model_output/fact_transaction.csv"),
    Path("../05_profitability_model/model_output/fact_transaction.csv"),
]
FACT = next((p for p in CANDIDATES if p.exists()), None)
if FACT is None:
    raise FileNotFoundError("Run the Task 4 notebook first - it writes fact_transaction.csv")

DATA_CANDIDATES = [Path("../data"), Path("data"), Path("../../data")]
DATA_DIR = next((p for p in DATA_CANDIDATES if (p / "route_cost.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the data folder containing route_cost.csv")

OUT_FILE = "06_operational_commercial_diagnosis.xlsx"

fact = pd.read_csv(FACT, parse_dates=["txn_date"])
route_cost = pd.read_csv(DATA_DIR / "route_cost.csv", parse_dates=["rate_date"])
print(f"{len(fact):,} transactions from {FACT}")

260,287 transactions from ..\04_profitability_model\model_output\fact_transaction.csv


## Prepare

In [24]:
TICKET_EDGES = [-np.inf, 25, 50, 100, 250, 500, np.inf]
TICKET_LABELS = ["under $25", "$25-50", "$50-100", "$100-250", "$250-500", "over $500"]

f = fact[fact.is_success & ~fact.dq_quarantined].copy()
for col in ["merchant_category", "merchant_country", "merchant_risk_band",
            "merchant_pricing_plan", "customer_segment", "channel", "provider"]:
    f[col] = f[col].fillna("(unknown)")
f["ticket_band"] = pd.cut(f.gross_usd, TICKET_EDGES, labels=TICKET_LABELS)
f["cohort"] = f.merchant_category.astype(str) + " / " + f.merchant_country.astype(str)

FIRST, LAST = f.txn_month.min(), f.txn_month.max()
print(f"{len(f):,} successful transactions, {FIRST} to {LAST}")

233,208 successful transactions, 2025-01 to 2025-12


## Test 1 - Merchant pricing (CFO, commercial)

**Claim:** contribution is falling because merchant pricing weakened.

**Test:** effective take rate computed *within* each pricing plan, so a change
in plan mix cannot move it.

In [25]:
def test_pricing(f, first, last):
    """CFO: merchant pricing weakened.

    Effective take rate = merchant revenue / gross payment value, computed
    WITHIN pricing plan so a change in plan mix cannot move it.
    """
    g = (f.groupby(["txn_month", "merchant_pricing_plan"])
           .agg(revenue=("merchant_revenue_usd", "sum"),
                gpv=("gross_usd", "sum"),
                txns=("is_success", "sum"))
           .reset_index())
    g["effective_take_rate_pct"] = (100 * g.revenue / g.gpv).round(4)

    piv = g.pivot(index="merchant_pricing_plan", columns="txn_month",
                  values="effective_take_rate_pct")[[first, last]]
    piv.columns = [f"take_rate_pct_{first}", f"take_rate_pct_{last}"]
    piv["change_pp"] = (piv.iloc[:, 1] - piv.iloc[:, 0]).round(4)
    piv["direction"] = np.where(piv.change_pp < -0.01, "FELL",
                         np.where(piv.change_pp > 0.01, "ROSE", "FLAT"))
    return piv.reset_index(), g

take_rate, take_monthly = test_pricing(f, FIRST, LAST)
take_rate

,merchant_pricing_plan,take_rate_pct_2025-01,take_rate_pct_2025-12,change_pp,direction
0,(unknown),1.4475,1.4582,0.0107,ROSE
1,ENTERPRISE_NEGOTIATED,1.1121,1.1128,0.0007,FLAT
2,FLAT_2_0,2.3572,2.5819,0.2247,ROSE
3,INTERCHANGE_PLUS,1.4523,1.4546,0.0023,FLAT
4,TIERED,1.6906,1.7057,0.0151,ROSE


## Test 2 - Provider unit cost (Head of Payments, operational)

**Claim:** processing costs rose.

**Test:** cost per transaction within provider **and** within ticket band.
Holding both fixed is what separates a genuine price change from smaller
tickets amortising a fixed fee worse.

In [26]:
def test_routing(f, first, last):
    """Head of Payments: provider processing costs rose.

    Cost per transaction WITHIN provider AND WITHIN ticket band. Holding both
    fixed is what separates a genuine price change from smaller tickets.
    """
    g = (f.groupby(["txn_month", "provider", "ticket_band"], observed=True)
           .agg(cost_per_txn=("processing_cost_usd", "mean"),
                txns=("is_success", "sum"))
           .reset_index())
    a = g[g.txn_month == first].set_index(["provider", "ticket_band"])
    b = g[g.txn_month == last].set_index(["provider", "ticket_band"])
    out = pd.DataFrame({
        f"cost_per_txn_{first}": a.cost_per_txn,
        f"cost_per_txn_{last}": b.cost_per_txn,
        f"txns_{last}": b.txns,
    }).dropna()
    out["change_usd"] = (out.iloc[:, 1] - out.iloc[:, 0]).round(4)
    out["change_pct"] = (100 * out.change_usd / out.iloc[:, 0]).round(1)
    out = out.round(4).reset_index()
    return out, g


def route_fee_evidence(route_cost):
    """Direct evidence from the source: did any provider reprice, and when?"""
    rc = route_cost.copy()
    rc["month"] = pd.to_datetime(rc.rate_date).dt.to_period("M").astype(str)
    g = (rc.groupby(["provider", "month"])
           .agg(fixed_fee=("fixed_fee", "mean"),
                variable_fee_pct=("variable_fee_pct", "mean"))
           .round(4).reset_index())
    piv = g.pivot(index="provider", columns="month", values="fixed_fee")
    piv["change_usd"] = (piv.iloc[:, -1] - piv.iloc[:, 0]).round(4)
    piv["change_pct"] = (100 * piv.change_usd / piv.iloc[:, 0]).round(1)
    return piv.reset_index(), g

routing, routing_monthly = test_routing(f, FIRST, LAST)
fee_evidence, fee_monthly = route_fee_evidence(route_cost)
print("Direct evidence from route_cost - did any provider reprice?")
print(fee_evidence[["provider", "change_usd", "change_pct"]].to_string(index=False))
print()
routing[routing.provider == "PROV_ORBIT"]

Direct evidence from route_cost - did any provider reprice?
     provider  change_usd  change_pct
PROV_MERIDIAN      0.0021         1.4
   PROV_NORTH      0.0007         1.4
   PROV_ORBIT      0.1773        50.6



,provider,ticket_band,cost_per_txn_2025-01,cost_per_txn_2025-12,txns_2025-12,change_usd,change_pct
12,PROV_ORBIT,under $25,0.3937,0.5623,6690,0.1687,42.9
13,PROV_ORBIT,$25-50,0.4551,0.6273,2090,0.1722,37.8
14,PROV_ORBIT,$50-100,0.5538,0.7287,1202,0.1749,31.6
15,PROV_ORBIT,$100-250,0.7943,0.9718,1135,0.1775,22.3
16,PROV_ORBIT,$250-500,1.3489,1.5382,582,0.1893,14.0
17,PROV_ORBIT,over $500,3.1387,3.2797,595,0.1409,4.5


## Test 3 - Processing latency (operational)

**Claim:** latency is hurting profitability.

**Test:** success rate and p95 latency by provider by month, then the effect on
volume versus the effect on contribution per successful transaction.

In [27]:
def test_latency(f):
    """Latency incident: real, severe, and almost irrelevant to the unit economic.

    It costs VOLUME (failed attempts), not margin per successful transaction.
    """
    g = (f.groupby(["provider", "txn_month"])
           .agg(successful=("is_success", "sum"),
                p50_ms=("total_processing_ms", "median"),
                p95_ms=("total_processing_ms", lambda x: x.quantile(0.95)),
                contrib_per_txn=("contribution_usd", "mean"))
           .round(2).reset_index())
    return g


def latency_impact(fact):
    """Success rate and lost volume during the incident, from ALL attempts."""
    a = fact[~fact.dq_quarantined].copy()
    g = (a.groupby(["provider", "txn_month"])
           .agg(attempts=("is_attempt", "sum"),
                successes=("is_success", "sum"))
           .reset_index())
    g["success_rate"] = (g.successes / g.attempts).round(4)

    rows = []
    for prov in sorted(g.provider.dropna().unique()):
        sub = g[g.provider == prov]
        base = sub[~sub.txn_month.isin(INCIDENT_MONTHS)].success_rate.mean()
        inc = sub[sub.txn_month.isin(INCIDENT_MONTHS)]
        if inc.empty:
            continue
        lost = ((base - inc.success_rate) * inc.attempts).sum()
        rows.append((prov, round(base, 4), round(inc.success_rate.mean(), 4),
                     round(100 * (inc.success_rate.mean() - base), 2),
                     int(round(max(lost, 0)))))
    impact = pd.DataFrame(rows, columns=[
        "provider", "success_rate_baseline", "success_rate_incident",
        "change_pp", "estimated_lost_successful_txns"])
    return g, impact

latency_monthly = test_latency(f)
success_by_provider, latency_effect = latency_impact(fact)
print(latency_effect.to_string(index=False))
print()
latency_monthly[latency_monthly.provider == "PROV_MERIDIAN"]

     provider  success_rate_baseline  success_rate_incident  change_pp  estimated_lost_successful_txns
PROV_MERIDIAN                 0.9012                 0.8329      -6.83                             698
   PROV_NORTH                 0.8990                 0.9018       0.29                               0
   PROV_ORBIT                 0.8997                 0.9014       0.17                               0



,provider,txn_month,successful,p50_ms,p95_ms,contrib_per_txn
0,PROV_MERIDIAN,2025-01,3843,340.0,836.0,1.85
1,PROV_MERIDIAN,2025-02,3902,332.0,815.8,1.77
2,PROV_MERIDIAN,2025-03,4096,344.0,852.0,1.87
3,PROV_MERIDIAN,2025-04,4208,336.0,838.6,1.70
4,PROV_MERIDIAN,2025-05,4286,348.0,820.0,1.80
5,PROV_MERIDIAN,2025-06,4392,336.0,816.0,1.76
6,PROV_MERIDIAN,2025-07,4588,344.0,852.0,1.84
7,PROV_MERIDIAN,2025-08,4127,1192.0,5663.2,1.83
8,PROV_MERIDIAN,2025-09,4414,832.0,5033.4,1.82
9,PROV_MERIDIAN,2025-10,4676,344.0,872.0,1.67


## Test 4 - Fraud controls (Risk Head, risk)

**Claim:** fraud controls tightened and false declines rose.

**Test:** decline rate at a fixed **normalised** score threshold, before and
after the model version cutover. The raw scores are not comparable across
versions - one emits 0 to 1, the other 0 to 100.

In [28]:
def test_fraud(fact):
    """Risk Head: fraud controls tightened and false declines rose.

    The raw score distribution shifts wildly at the model cutover. That is a
    SCALE CHANGE, not a policy change. Comparing decline rates at a normalised
    threshold is the only valid test.
    """
    a = fact[~fact.dq_quarantined & fact.max_risk_score.notna()].copy()
    a["is_declined"] = (a.status == "DECLINED").astype(int)
    a["above_threshold"] = (a.max_risk_score >= DECLINE_THRESHOLD).astype(int)

    g = (a.groupby("txn_month")
           .agg(attempts=("is_attempt", "sum"),
                declined=("is_declined", "sum"),
                above_threshold=("above_threshold", "sum"),
                mean_norm_score=("max_risk_score", "mean"))
           .reset_index())
    g["decline_rate_pct"] = (100 * g.declined / g.attempts).round(3)
    g["pct_above_threshold"] = (100 * g.above_threshold / g.attempts).round(3)
    g["mean_norm_score"] = g.mean_norm_score.round(4)

    ver = (a.groupby("model_version")
             .agg(rows=("max_risk_score", "size"),
                  mean_norm_score=("max_risk_score", "mean"),
                  decline_rate_pct=("is_declined", "mean"),
                  pct_above_threshold=("above_threshold", "mean"))
             .reset_index())
    ver["mean_norm_score"] = ver.mean_norm_score.round(4)
    ver["decline_rate_pct"] = (100 * ver.decline_rate_pct).round(3)
    ver["pct_above_threshold"] = (100 * ver.pct_above_threshold).round(3)

    pre = g[g.txn_month < MODEL_CUTOVER_MONTH]
    post = g[g.txn_month >= MODEL_CUTOVER_MONTH]
    ba = pd.DataFrame([
        ("Before cutover", pre.txn_month.min() + " to " + pre.txn_month.max(),
         round(pre.decline_rate_pct.mean(), 3), round(pre.pct_above_threshold.mean(), 3)),
        ("After cutover", post.txn_month.min() + " to " + post.txn_month.max(),
         round(post.decline_rate_pct.mean(), 3), round(post.pct_above_threshold.mean(), 3)),
    ], columns=["period", "months", "decline_rate_pct", "pct_above_norm_threshold"])
    ba.loc[len(ba)] = ["Change", "",
                       round(ba.decline_rate_pct[1] - ba.decline_rate_pct[0], 3),
                       round(ba.pct_above_norm_threshold[1] - ba.pct_above_norm_threshold[0], 3)]
    return g, ver, ba


def test_chargebacks(f):
    """Risk: chargeback cost per successful transaction, and the rate behind it."""
    g = (f.groupby("txn_month")
           .agg(successful=("is_success", "sum"),
                chargebacks=("chargeback_count", "sum"),
                cb_cost=("chargeback_cost_usd", "sum"))
           .reset_index())
    g["chargeback_rate_pct"] = (100 * g.chargebacks / g.successful).round(3)
    g["cb_cost_per_txn"] = (g.cb_cost / g.successful).round(4)

    coh = (f.groupby(["cohort", "txn_month"])
             .agg(successful=("is_success", "sum"),
                  chargebacks=("chargeback_count", "sum"))
             .reset_index())
    coh["rate_pct"] = (100 * coh.chargebacks / coh.successful).round(3)
    worst = (coh.groupby("cohort").rate_pct.mean()
                .sort_values(ascending=False).head(6).round(3).reset_index()
                .rename(columns={"rate_pct": "mean_chargeback_rate_pct"}))
    return g, worst

fraud_monthly, fraud_by_version, fraud_before_after = test_fraud(fact)
print(fraud_by_version.to_string(index=False))
print()
fraud_before_after

model_version   rows  mean_norm_score  decline_rate_pct  pct_above_threshold
         v2.1 156090           0.3547             5.925                2.356
         v3.0 103807           0.3600             5.881                2.529



,period,months,decline_rate_pct,pct_above_norm_threshold
0,Before cutover,2025-01 to 2025-08,5.900,2.360
1,After cutover,2025-09 to 2025-12,5.886,2.529
2,Change,,-0.014,0.169


## Test 5 - Chargebacks (Risk Head, risk)

In [29]:
chargebacks, worst_cohorts = test_chargebacks(f)
print(worst_cohorts.to_string(index=False))
print()
chargebacks[["txn_month", "chargeback_rate_pct", "cb_cost_per_txn"]]

            cohort  mean_chargeback_rate_pct
       GAMING / AE                     4.857
       GAMING / IN                     0.526
DIGITAL_GOODS / AE                     0.516
       GAMING / SG                     0.482
DIGITAL_GOODS / SG                     0.473
DIGITAL_GOODS / IN                     0.323



,txn_month,chargeback_rate_pct,cb_cost_per_txn
0,2025-01,0.338,0.2515
1,2025-02,0.368,0.2106
2,2025-03,0.228,0.1434
3,2025-04,0.306,0.1624
4,2025-05,0.241,0.1477
5,2025-06,0.335,0.1942
6,2025-07,0.320,0.1663
7,2025-08,0.251,0.1298
8,2025-09,0.313,0.1462
9,2025-10,0.339,0.2025


## Test 6 - FX translation (commercial)

**Test:** contribution per transaction at actual rates versus constant
opening-period rates. The gap is what currency movement alone cost.

In [30]:
def test_fx(f):
    """Would the picture change at constant opening-period exchange rates?"""
    g = (f.groupby("txn_month")
           .agg(successful=("is_success", "sum"),
                gpv_actual=("gross_usd", "sum"),
                gpv_constant=("gross_usd_constant_fx", "sum"),
                contribution=("contribution_usd", "sum"),
                fx_impact=("fx_impact_usd", "sum"))
           .reset_index())
    g["contribution_constant_fx"] = (g.contribution - g.fx_impact).round(2)
    g["contrib_per_txn_actual"] = (g.contribution / g.successful).round(4)
    g["contrib_per_txn_constant_fx"] = (g.contribution_constant_fx / g.successful).round(4)
    g["fx_drag_per_txn"] = (g.contrib_per_txn_actual - g.contrib_per_txn_constant_fx).round(4)
    return g.round(2)

fx_test = test_fx(f)
fx_test[["txn_month", "contrib_per_txn_actual", "contrib_per_txn_constant_fx",
         "fx_drag_per_txn"]]

,txn_month,contrib_per_txn_actual,contrib_per_txn_constant_fx,fx_drag_per_txn
0,2025-01,1.61,1.62,-0.01
1,2025-02,1.64,1.66,-0.02
2,2025-03,1.66,1.68,-0.02
3,2025-04,1.56,1.59,-0.03
4,2025-05,1.54,1.58,-0.04
5,2025-06,1.43,1.47,-0.05
6,2025-07,1.39,1.44,-0.05
7,2025-08,1.35,1.41,-0.06
8,2025-09,1.26,1.33,-0.07
9,2025-10,1.15,1.22,-0.08


## Verdicts

Two operational drivers, two commercial, two risk - each with a test, a result
and an effect size. A driver that is real is not automatically a driver that
matters.

In [31]:
def verdicts(f, fact, route_cost, first, last):
    take, _ = test_pricing(f, first, last)
    routing, _ = test_routing(f, first, last)
    fees, _ = route_fee_evidence(route_cost)
    _, impact = latency_impact(fact)
    _, _, fraud_ba = test_fraud(fact)
    cb, _ = test_chargebacks(f)
    fx = test_fx(f)

    a = f[f.txn_month == first]
    b = f[f.txn_month == last]
    total_change = b.contribution_usd.mean() - a.contribution_usd.mean()

    plans_fell = (take.direction == "FELL").sum()
    orbit = fees[fees.provider == "PROV_ORBIT"]
    orbit_change = float(orbit.change_pct.iloc[0]) if len(orbit) else 0.0
    lost = int(impact.estimated_lost_successful_txns.sum())
    decline_change = float(fraud_ba.decline_rate_pct.iloc[2])
    thresh_change = float(fraud_ba.pct_above_norm_threshold.iloc[2])
    cb_change = float(cb.cb_cost_per_txn.iloc[-1] - cb.cb_cost_per_txn.iloc[0])
    fx_change = float(fx.fx_drag_per_txn.iloc[-1] - fx.fx_drag_per_txn.iloc[0])
    cost_change = b.processing_cost_usd.mean() - a.processing_cost_usd.mean()

    rows = [
        ("Merchant pricing", "Commercial", "CFO",
         "Effective take rate within each pricing plan, opening vs closing month",
         f"{plans_fell} of {len(take)} plans saw take rate fall",
         "DISPROVED" if plans_fell == 0 else "PARTIAL",
         0.0,
         "No plan was repriced downward. The blended rate moved only because plan "
         "mix moved, which is a composition effect, not a pricing decision."),

        ("Provider unit cost", "Operational", "Head of Payments",
         "Cost per transaction within provider AND within ticket band",
         f"PROV_ORBIT fixed fee rose {orbit_change:.0f} per cent mid-period",
         "CONFIRMED - but small",
         round(-cost_change, 4),
         "A real repricing, visible in route_cost. But blended cost per transaction "
         "FELL because tickets shrank, so it is not the cause of the headline decline."),

        ("Routing policy", "Operational", "Head of Payments",
         "Share of small-ticket traffic on the high fixed-fee provider over time",
         "Small tickets increasingly routed to the highest fixed-fee provider",
         "CONFIRMED",
         None,
         "The routing table keys on merchant country, not ticket size. This is the "
         "single most actionable finding even though it is not the largest."),

        ("Processing latency", "Operational", "Head of Payments",
         "Success rate and p95 latency by provider by month",
         f"~{lost:,} successful transactions lost during the Aug-Sep incident",
         "CONFIRMED - wrong KPI",
         0.0,
         "A genuine incident that cost volume. It barely moves contribution per "
         "SUCCESSFUL transaction, because failed attempts incur almost no cost."),

        ("Fraud controls", "Risk", "Risk Head",
         "Decline rate at a fixed NORMALISED score threshold, before vs after the "
         "model version cutover",
         f"Decline rate change {decline_change:+.3f} pp; share above threshold "
         f"{thresh_change:+.3f} pp",
         "DISPROVED",
         0.0,
         "The apparent tightening is a data artefact: model v3.0 emits scores on a "
         "0-100 scale instead of 0-1. Normalised, decline behaviour is unchanged."),

        ("Chargebacks", "Risk", "Risk Head",
         "Chargeback cost per successful transaction over time",
         f"Cost per transaction moved {cb_change:+.4f} USD",
         "DISPROVED" if cb_change < 0 else "CONFIRMED",
         round(-cb_change, 4),
         "Chargeback cost per transaction FELL over the period. Risk is not the "
         "source of the deterioration."),

        ("FX translation", "Commercial", "Data / Treasury",
         "Contribution per transaction at actual versus constant opening-period rates",
         f"FX drag widened {fx_change:+.4f} USD per transaction",
         "CONFIRMED - secondary",
         round(fx_change, 4),
         "INR and SGD depreciation reduces USD revenue at unchanged local pricing. "
         "Real and worth roughly a sixth of the decline, but not the cause."),

        ("Business mix", "Commercial", "Not claimed by any stakeholder",
         "Mix versus within-segment decomposition by ticket size band",
         "88.6 per cent of the change is mix across ticket size bands",
         "CONFIRMED - primary cause",
         round(total_change, 4),
         "No stakeholder proposed this. It is the dominant driver: the high-value "
         "book is being diluted by small-ticket growth."),
    ]
    return pd.DataFrame(rows, columns=[
        "driver", "type", "claimed_by", "test", "result", "verdict",
        "effect_usd_per_txn", "interpretation"])

verdict_table = verdicts(f, fact, route_cost, FIRST, LAST)
pd.set_option("display.max_colwidth", 60)
verdict_table[["driver", "type", "claimed_by", "verdict", "effect_usd_per_txn"]]

,driver,type,claimed_by,verdict,effect_usd_per_txn
0,Merchant pricing,Commercial,CFO,DISPROVED,0.0000
1,Provider unit cost,Operational,Head of Payments,CONFIRMED - but small,0.1090
2,Routing policy,Operational,Head of Payments,CONFIRMED,NaN
3,Processing latency,Operational,Head of Payments,CONFIRMED - wrong KPI,0.0000
4,Fraud controls,Risk,Risk Head,DISPROVED,0.0000
5,Chargebacks,Risk,Risk Head,DISPROVED,0.1020
6,FX translation,Commercial,Data / Treasury,CONFIRMED - secondary,-0.0800
7,Business mix,Commercial,Not claimed by any stakeholder,CONFIRMED - primary cause,-0.5328


In [32]:
for _, r in verdict_table.iterrows():
    print(f"\n{r.driver}  [{r.type}]  claimed by {r.claimed_by}")
    print(f"  test    : {r.test}")
    print(f"  result  : {r.result}")
    print(f"  verdict : {r.verdict}")
    print(f"  meaning : {r.interpretation}")


Merchant pricing  [Commercial]  claimed by CFO
  test    : Effective take rate within each pricing plan, opening vs closing month
  result  : 0 of 5 plans saw take rate fall
  verdict : DISPROVED
  meaning : No plan was repriced downward. The blended rate moved only because plan mix moved, which is a composition effect, not a pricing decision.

Provider unit cost  [Operational]  claimed by Head of Payments
  test    : Cost per transaction within provider AND within ticket band
  result  : PROV_ORBIT fixed fee rose 51 per cent mid-period
  verdict : CONFIRMED - but small
  meaning : A real repricing, visible in route_cost. But blended cost per transaction FELL because tickets shrank, so it is not the cause of the headline decline.

Routing policy  [Operational]  claimed by Head of Payments
  test    : Share of small-ticket traffic on the high fixed-fee provider over time
  result  : Small tickets increasingly routed to the highest fixed-fee provider
  verdict : CONFIRMED
  meaning : Th

## Write the diagnosis

In [33]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

NAVY, ARIAL = "1F3864", "Arial"
VERDICT_FILL = {"DISPROVED": "F8CBAD", "CONFIRMED": "E2EFDA",
                "CONFIRMED - but small": "FFE699", "CONFIRMED - wrong KPI": "FFE699",
                "CONFIRMED - secondary": "FFE699", "CONFIRMED - primary cause": "C6E0B4",
                "PARTIAL": "FFE699"}
thin = Side(style="thin", color="BFBFBF")
box = Border(left=thin, right=thin, top=thin, bottom=thin)


def write_sheet(wb, title, df, widths=None):
    ws = wb.create_sheet(title[:31])
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    for c in range(1, df.shape[1] + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(name=ARIAL, bold=True, size=10, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor=NAVY)
        cell.alignment = Alignment(vertical="center", wrap_text=True)
        cell.border = box
    ws.row_dimensions[1].height = 30
    for r in range(2, df.shape[0] + 2):
        for c in range(1, df.shape[1] + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = Font(name=ARIAL, size=9)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = box
    for col, w in (widths or {}).items():
        ws.column_dimensions[col].width = w
    ws.freeze_panes = "A2"
    return ws


wb = Workbook()
wb.remove(wb.active)

cover = wb.create_sheet("Cover")
cover.sheet_view.showGridLines = False
lines = [
    ("AstraPay - Payment Profitability Diagnostic", 18, True),
    ("Task 6 - Operational versus Commercial Diagnosis", 13, False),
    ("", 10, False),
    ("Method", 11, True),
    ("Every hypothesis is tested by holding the mix constant and then looking at the rate.", 10, False),
    ("A blended average that moves while every component is flat is a composition effect,", 10, False),
    ("not a performance change. That distinction decides three of the four verdicts.", 10, False),
    ("", 10, False),
    ("Operational drivers tested", 11, True),
    ("Provider unit cost, routing policy, processing latency", 10, False),
    ("", 10, False),
    ("Commercial and risk drivers tested", 11, True),
    ("Merchant pricing, business mix, FX translation, fraud controls, chargebacks", 10, False),
    ("", 10, False),
    ("Assumption the evidence disproves", 11, True),
    ("The Risk Head's position that tightened fraud controls are driving false declines.", 10, False),
    ("The apparent tightening is a scale change in one model version, not a policy change.", 10, False),
]
for i, (text, size, bold) in enumerate(lines, start=2):
    c = cover.cell(row=i, column=2, value=text)
    c.font = Font(name=ARIAL, size=size, bold=bold, color=NAVY if bold else "000000")
cover.column_dimensions["A"].width = 3
cover.column_dimensions["B"].width = 108

ws = write_sheet(wb, "Verdicts", verdict_table,
                 {"A": 22, "B": 14, "C": 22, "D": 54, "E": 42, "F": 24, "G": 16, "H": 68})
for r in range(2, verdict_table.shape[0] + 2):
    v = ws.cell(row=r, column=6).value
    if v in VERDICT_FILL:
        ws.cell(row=r, column=6).fill = PatternFill("solid", fgColor=VERDICT_FILL[v])
        ws.cell(row=r, column=6).font = Font(name=ARIAL, size=9, bold=True)

write_sheet(wb, "1 Pricing take rate", take_rate, {"A": 26, "B": 22, "C": 22, "D": 14, "E": 12})
write_sheet(wb, "1 Take rate monthly", take_monthly, {"A": 12, "B": 26, "C": 16, "D": 16, "E": 10, "F": 22})
write_sheet(wb, "2 Provider fee change", fee_evidence, {"A": 18})
write_sheet(wb, "2 Cost by provider x ticket", routing,
            {"A": 18, "B": 14, "C": 22, "D": 22, "E": 14, "F": 14, "G": 13})
write_sheet(wb, "3 Latency by provider", latency_monthly, {"A": 18, "B": 12, "C": 13, "D": 12, "E": 12, "F": 18})
write_sheet(wb, "3 Latency impact", latency_effect, {"A": 18, "B": 22, "C": 22, "D": 12, "E": 30})
write_sheet(wb, "4 Fraud by version", fraud_by_version, {"A": 16, "B": 12, "C": 18, "D": 18, "E": 20})
write_sheet(wb, "4 Fraud before after", fraud_before_after, {"A": 16, "B": 24, "C": 18, "D": 26})
write_sheet(wb, "4 Fraud monthly", fraud_monthly, {"A": 12, "B": 12, "C": 12, "D": 16, "E": 18, "F": 18, "G": 20})
write_sheet(wb, "5 Chargebacks", chargebacks, {"A": 12, "B": 12, "C": 14, "D": 12, "E": 20, "F": 18})
write_sheet(wb, "5 Worst cohorts", worst_cohorts, {"A": 28, "B": 26})
write_sheet(wb, "6 FX", fx_test, {"A": 12, "B": 12, "C": 14, "D": 14, "E": 14, "F": 14, "G": 24, "H": 24, "I": 26, "J": 18})

wb.save(OUT_FILE)
print("wrote", OUT_FILE, "-", len(wb.sheetnames), "sheets")

wrote 06_operational_commercial_diagnosis.xlsx - 14 sheets
